# FEATURE TRANSFORMATION

Feature transformation means **changing the mathematical representation of a feature** to make it more suitable for analysis or machine learning.

### Common Goals

- Reduce skewness
- Compress extreme values
- Stabilize variance
- Make distributions more suitable for certain ML/statistical methods
- Improve the behavior of some machine learning models

---

# 1. LOG TRANSFORMATION

Log transformation **strongly compresses large values**.

It is especially useful for **strongly right-skewed data**.

### Common Examples

- Salary
- Income
- House prices
- Population
- Transaction values

### Python

```python
import numpy as np

df["Salary_log"] = np.log1p(df["Salary"])
````

`np.log1p(x)` calculates:

```text
log(1 + x)
```

Using `log1p()` is useful because it can safely handle `0`:

```text
log1p(0) = log(1) = 0
```

### Effect

Example:

```text
Original:
100
1,000
10,000
100,000
```

After log transformation, the large differences are compressed.

```text
Small values → changed less
Large values → compressed more
```

Log transformation is useful when **large values dominate the distribution**.

---

# 2. SQUARE ROOT TRANSFORMATION

Square root transformation reduces skewness **more mildly than log transformation**.

### Python

```python
df["Salary_sqrt"] = np.sqrt(df["Salary"])
```

### Useful For

* Moderately skewed data
* Count data
* Positive numerical features

### General Comparison

```text
Log
↓
Stronger compression

Square Root
↓
Milder compression
```

---

# 3. BOX-COX TRANSFORMATION

**Box-Cox** is a family of **power transformations**.

It finds a suitable transformation parameter called **lambda (λ)** to transform the distribution.

### Important Requirement

Classic Box-Cox requires **strictly positive values**.

```text
10   ✓
20   ✓
0    ✗
-5   ✗
```

### Python

```python
from scipy.stats import boxcox

df["Salary_boxcox"], lambda_value = boxcox(
    df["Salary"]
)
```

The function returns:

1. Transformed values
2. The lambda value used for the transformation

### Important

> **Box-Cox → Positive values only**

It cannot directly handle:

* Zero
* Negative values

---

# 4. YEO-JOHNSON TRANSFORMATION

**Yeo-Johnson** is another power transformation.

Its major advantage is that it can handle:

* Positive values
* Zero
* Negative values

Example:

```text
-100
-20
0
10
50
```

Yeo-Johnson can handle all of these values.

### Python

```python
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(
    method="yeo-johnson"
)

df["Salary_yeojohnson"] = pt.fit_transform(
    df[["Salary"]]
)
```

The transformation can make a highly skewed distribution more symmetric.

---

# BOX-COX vs YEO-JOHNSON

| Feature          | Box-Cox              | Yeo-Johnson          |
| ---------------- | -------------------- | -------------------- |
| Positive values  | ✓                    | ✓                    |
| Zero values      | ✗                    | ✓                    |
| Negative values  | ✗                    | ✓                    |
| Type             | Power transformation | Power transformation |
| Lambda parameter | ✓                    | ✓                    |

### Simple Rule

```text
Only positive values
        ↓
    Box-Cox
```

```text
Positive + Zero + Negative
        ↓
   Yeo-Johnson
```

---

# 5. CHECK SKEWNESS BEFORE TRANSFORMATION

First check the skewness of the feature:

```python
df["Salary"].skew()
```

### General Interpretation

```text
Skewness ≈ 0
↓
Approximately symmetric
```

```text
Large positive skewness
↓
Right-skewed
```

```text
Large negative skewness
↓
Left-skewed
```

There is no universal cutoff that determines when transformation is mandatory. The distribution, model, and problem should all be considered.

---

# TRANSFORMATION DECISION

If the feature is approximately symmetric:

```text
No transformation may be necessary.
```

If the feature is strongly right-skewed:

Consider:

* Log transformation
* Square root transformation
* Box-Cox transformation
* Yeo-Johnson transformation

If zero or negative values are present:

Consider:

* Yeo-Johnson

If the data is strictly positive:

Consider:

* Log
* Square root
* Box-Cox

---

# IMPORTANT

Do **not** transform features blindly.

Before transforming:

1. Check the distribution.
2. Check skewness.
3. Understand the meaning of the feature.
4. Check whether the feature contains zero or negative values.
5. Choose an appropriate transformation.
6. Compare the distribution before and after transformation.
7. Check whether the transformation actually improves the downstream analysis or ML model.

---

# QUICK MEMORY 🧠

```text
Log
→ Strong compression
→ Useful for strongly right-skewed positive data
```

```text
Square Root
→ Mild compression
→ Useful for moderately skewed / count data
```

```text
Box-Cox
→ Power transformation
→ Positive values only
```

```text
Yeo-Johnson
→ Power transformation
→ Handles positive, zero, and negative values
```

### Key Idea

> **Feature Transformation = Change the mathematical representation of a feature without changing the underlying meaning of the data.**

```
```


In [1]:

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Salary": [
        20000, 22000, 25000, 27000,
        30000, 35000, 40000, 50000,
        70000, 100000, 200000, 500000
    ],

    "Transaction_Count": [
        1, 2, 2, 3,
        4, 5, 7, 10,
        15, 20, 30, 50
    ],

    "Temperature": [
        -10, -5, 0, 2,
        5, 10, 15, 20,
        25, 30, 35, 40
    ]
})

In [7]:
df.skew(numeric_only = True)

Salary               2.767728
Transaction_Count    1.820758
Temperature          0.184530
dtype: float64

In [8]:
df["Salary_log"] = np.log1p(df["Salary"])

In [9]:
print(df["Salary_log"].skew())
print(df["Salary"].skew())

1.3401587627778302
2.767728042509286


In [10]:
df["Transaction_sqrt"] = np.sqrt(
    df["Transaction_Count"]
)

In [12]:
print(df["Transaction_Count"].skew())
print(df["Transaction_sqrt"].skew())

1.8207584747367123
1.0588406283895975


In [13]:
from scipy.stats import boxcox

df["Salary_boxcox"], lam = boxcox(
    df["Salary"]
)

print("Lambda:", lam)

Lambda: -0.7217531652027996


In [14]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(
    method="yeo-johnson"
)

df["Temperature_yeojohnson"] = pt.fit_transform(
    df[["Temperature"]]
)

In [16]:
print(df["Temperature"].skew())
print(df["Temperature_yeojohnson"].skew())

0.1845304298931565
-0.19210830838449022


In [17]:
# for the temperature even in normal the skewness didnt have very much lil bit right skewed
# using yeojhonson the skewness is gone lil bit left side

In [29]:
print(df["Salary"].skew())
print(df["Salary_log"].skew())
print(df["Salary_boxcox"].skew())

2.767728042509286
1.3401587627778302
0.35276866174784655


In [19]:
# after using log tansfromation for salary its still lil bit right skewed
# after using boxcox the values close to normal distribution

In [26]:
print(df["Transaction_Count"].skew())
print(df["Transaction_sqrt"].skew())
df['Transaction_log'] = np.log1p(df["Transaction_Count"])
print(df["Transaction_log"].skew())

1.8207584747367123
1.0588406283895975
0.41342556866038044


In [27]:
# the use of sqrt transformation reduced skewness but it stil greater than 1
# log transformation reduced it very well

In [ ]:
# 1)  for salary we can go with boxcox or log transfromation
# 2) temperature we can keep it as it as because it doesnt have very much of skeweness
# 3) for transformation count  we can use the log transformation because it reducs it very much 